# NOVALAINE — Mission d’audit qualité des données
**Responsable : AKBI Hayat — Auditrice qualité**

## Objectif
Auditer `clients.csv`, `commandes.csv`, `lignes_commande.csv` et `produits.csv`, mesurer l’impact des corrections techniques et contrôler la cohérence inter-tables ainsi que la réconciliation du chiffre d’affaires.

**Règles de travail**
- Les fichiers bruts ne sont jamais modifiés.
- La date de référence est fixée au **30/09/2026** (pas de `today()`).
- Une anomalie n’est pas automatiquement supprimée : les cas incertains sont signalés et documentés.
- Les corrections sont appliquées sur des copies et leur impact est chiffré.
- Les montants sont en euros TTC.


## 1. Préparation et chargement des données
Placez les quatre CSV dans le même dossier que ce notebook, ou dans un sous-dossier `donnees/`. Le code cherche aussi les fichiers dans `/mnt/data` lorsqu’il est exécuté dans l’environnement fourni.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
import matplotlib.pyplot as plt
from IPython.display import display

DATE_REFERENCE = pd.Timestamp("2026-09-30")
FRAIS_LIVRAISON = 4.90
SEUIL_LIVRAISON_GRATUITE = 90.00
TOLERANCE_EURO = 0.02

FICHIERS = ["clients.csv", "commandes.csv", "lignes_commande.csv", "produits.csv"]

def trouver_dossier_donnees():
    candidats = [Path("donnees"), Path("."), Path("/mnt/data")]
    for dossier in candidats:
        if all((dossier / nom).exists() for nom in FICHIERS):
            return dossier
    raise FileNotFoundError(
        "Fichiers introuvables. Placez les 4 CSV dans le dossier du notebook "
        "ou dans un sous-dossier 'donnees/'."
    )

DOSSIER_DONNEES = trouver_dossier_donnees()
brut = {
    "clients": pd.read_csv(DOSSIER_DONNEES / "clients.csv", dtype=str),
    "commandes": pd.read_csv(DOSSIER_DONNEES / "commandes.csv", dtype=str),
    "lignes": pd.read_csv(DOSSIER_DONNEES / "lignes_commande.csv", dtype=str),
    "produits": pd.read_csv(DOSSIER_DONNEES / "produits.csv", dtype=str),
}

# Copies de travail : les objets bruts ci-dessus restent intacts.
travail = {nom: df.copy(deep=True) for nom, df in brut.items()}

print(f"Dossier de données : {DOSSIER_DONNEES.resolve()}")
for nom, df in brut.items():
    print(f"{nom:12s} : {len(df):,} lignes | {df.shape[1]} colonnes")


Dossier de données : /mnt/data
clients      : 4,160 lignes | 13 colonnes
commandes    : 9,583 lignes | 9 colonnes
lignes       : 16,868 lignes | 5 colonnes
produits     : 120 lignes | 8 colonnes


## 2. Audit initial — structure, complétude et unicité

In [2]:
# Volumétries et types
profil_structure = pd.DataFrame([
    {"table": nom, "lignes": len(df), "colonnes": len(df.columns),
     "doublons_exacts": int(df.duplicated().sum())}
    for nom, df in brut.items()
])
display(profil_structure)

# Valeurs manquantes : nombre et pourcentage par colonne
lignes_manquants = []
for nom, df in brut.items():
    for col in df.columns:
        n = int(df[col].isna().sum() + df[col].astype("string").str.strip().eq("").sum())
        if n:
            lignes_manquants.append({
                "table": nom, "colonne": col, "valeurs_manquantes": n,
                "pct_lignes": round(100*n/len(df), 2) if len(df) else 0
            })
audit_manquants = pd.DataFrame(lignes_manquants).sort_values(
    ["table", "valeurs_manquantes"], ascending=[True, False]
)
display(audit_manquants)


,table,lignes,colonnes,doublons_exacts
0,clients,4160,13,0
1,commandes,9583,9,86
2,lignes,16868,5,81
3,produits,120,8,0


,table,colonne,valeurs_manquantes,pct_lignes
2,clients,date_naissance,393,9.45
0,clients,civilite,249,5.99
1,clients,email,212,5.10
4,clients,opt_in_email,163,3.92
3,clients,ville,49,1.18
6,commandes,magasin_id,7473,77.98
8,commandes,code_promo,7260,75.76
7,commandes,mode_livraison,2110,22.02
9,commandes,montant_ttc,192,2.00
5,commandes,client_id,47,0.49


## 3. Audit des clés et de l’intégrité référentielle

In [3]:
c, o, l, p = (brut["clients"], brut["commandes"], brut["lignes"], brut["produits"])

def valeurs_non_vides(df, col):
    return set(df[col].dropna().astype(str).str.strip()) - {""}

cles = {
    "client_id": valeurs_non_vides(c, "client_id"),
    "commande_id": valeurs_non_vides(o, "commande_id"),
    "produit_id": valeurs_non_vides(p, "produit_id"),
}
controle_cles = pd.DataFrame([
    {"contrôle": "client_id dupliqués dans clients", "anomalies": int(c["client_id"].duplicated(keep=False).sum())},
    {"contrôle": "commande_id dupliqués dans commandes", "anomalies": int(o["commande_id"].duplicated(keep=False).sum())},
    {"contrôle": "produit_id dupliqués dans produits", "anomalies": int(p["produit_id"].duplicated(keep=False).sum())},
    {"contrôle": "commandes sans client_id", "anomalies": int(o["client_id"].isna().sum() + o["client_id"].astype("string").str.strip().eq("").sum())},
    {"contrôle": "commandes avec client_id inconnu", "anomalies": int((~o["client_id"].fillna("").str.strip().isin(cles["client_id"]) & o["client_id"].fillna("").str.strip().ne("")).sum())},
    {"contrôle": "lignes avec commande_id inconnu", "anomalies": int((~l["commande_id"].fillna("").str.strip().isin(cles["commande_id"])).sum())},
    {"contrôle": "lignes avec produit_id inconnu", "anomalies": int((~l["produit_id"].fillna("").str.strip().isin(cles["produit_id"])).sum())},
])
display(controle_cles)

# Doublons d'e-mails : indicateur de rapprochement, pas preuve suffisante d'identité.
email_norm = c["email"].fillna("").str.strip().str.lower()
email_norm = email_norm.mask(email_norm.eq(""))
emails_repetes = c[email_norm.isin(email_norm[email_norm.duplicated(keep=False)])].copy()
print("Groupes d'e-mails répétés :", int(email_norm[email_norm.duplicated(keep=False)].nunique()))
display(emails_repetes[["client_id", "prenom", "nom", "email", "date_naissance", "source_systeme"]].head(20))


,contrôle,anomalies
0,client_id dupliqués dans clients,0
1,commande_id dupliqués dans commandes,180
2,produit_id dupliqués dans produits,0
3,commandes sans client_id,47
4,commandes avec client_id inconnu,116
5,lignes avec commande_id inconnu,0
6,lignes avec produit_id inconnu,102


Groupes d'e-mails répétés : 149


,client_id,prenom,nom,email,date_naissance,source_systeme
2,C000003,Paul,Martin,NaN,1974-09-02,web
8,C000009,Romain,Lambert,NaN,1972-08-24,web
22,C000023,Arthur,Leroy,arthur.leroy41@outlook.fr,1973-06-10,web
31,C000032,Mathieu,Martinez,NaN,1967-03-06,web
35,C000036,Marie,Lefèvre,marie.lefevre195@yahoo.fr,1951-04-24,web
43,C000044,Camille,Girard,camille.girard702@orange.fr,2003-01-01,web
59,C000060,Laura,Garcia,NaN,14/08/1977,boutique
60,C000061,Camille,Bertrand,camille.bertrand924@gmail.com,04/06/1955,web
75,C000076,Thomas,Thomas,NaN,1850-03-03,web
91,C000092,Thomas,Simon,NaN,18/10/1950,web


## 4. Audit de validité — dates, montants, quantités et statuts

In [4]:
def parse_montant(serie):
    s = serie.astype("string").str.replace("€", "", regex=False).str.replace("\\u202f", "", regex=False).str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def parse_date(serie):
    # Parsing mixte pour accepter les formats rencontrés ; les dates ambiguës restent à examiner.
    return pd.to_datetime(serie, errors="coerce", format="mixed", dayfirst=True)

cmd = o.copy()
cmd["date_parsee"] = parse_date(cmd["date_commande"])
cmd["montant_parse"] = parse_montant(cmd["montant_ttc"])
cl = c.copy()
cl["inscription_parsee"] = parse_date(cl["date_inscription"])
cl["naissance_parsee"] = parse_date(cl["date_naissance"])
lig = l.copy()
lig["quantite_num"] = pd.to_numeric(lig["quantite"], errors="coerce")
lig["prix_num"] = parse_montant(lig["prix_unitaire"])
lig["remise_num"] = parse_montant(lig["remise_pct"])

statuts = (o["statut"].fillna("").astype(str).str.strip().str.lower()
           .map(lambda x: ''.join(ch for ch in unicodedata.normalize("NFKD", x)
                                  if not unicodedata.combining(ch))))
audit_validite = pd.DataFrame([
    {"anomalie": "Dates de commande non interprétables", "nombre": int(cmd["date_parsee"].isna().sum())},
    {"anomalie": "Commandes postérieures au 30/09/2026", "nombre": int((cmd["date_parsee"] > DATE_REFERENCE).sum())},
    {"anomalie": "Dates d'inscription postérieures au 30/09/2026", "nombre": int((cl["inscription_parsee"] > DATE_REFERENCE).sum())},
    {"anomalie": "Montants de commande non numériques", "nombre": int(cmd["montant_parse"].isna().sum())},
    {"anomalie": "Montants de commande négatifs", "nombre": int((cmd["montant_parse"] < 0).sum())},
    {"anomalie": "Quantités nulles ou négatives", "nombre": int((lig["quantite_num"] <= 0).sum())},
    {"anomalie": "Prix unitaires non numériques", "nombre": int(lig["prix_num"].isna().sum())},
    {"anomalie": "Remises hors intervalle 0-100", "nombre": int((~lig["remise_num"].between(0, 100)).sum())},
    {"anomalie": "Dates de naissance futures (interprétation retenue)", "nombre": int((cl["naissance_parsee"] > DATE_REFERENCE).sum())},
    {"anomalie": "Statuts hors référentiel attendu", "nombre": int((~statuts.isin(["livree", "annulee", "retournee"])).sum())},
])
display(audit_validite)
print("Statuts normalisés et effectifs :")
display(statuts.value_counts(dropna=False).rename_axis("statut_normalise").reset_index(name="nombre"))


,anomalie,nombre
0,Dates de commande non interprétables,0
1,Commandes postérieures au 30/09/2026,47
2,Dates d'inscription postérieures au 30/09/2026,7
3,Montants de commande non numériques,192
4,Montants de commande négatifs,28
5,Quantités nulles ou négatives,51
6,Prix unitaires non numériques,166
7,Remises hors intervalle 0-100,0
8,Dates de naissance futures (interprétation ret...,102
9,Statuts hors référentiel attendu,0


Statuts normalisés et effectifs :


,statut_normalise,nombre
0,livree,8405
1,retournee,708
2,annulee,470


## 5. Mesure des corrections techniques
Les seules corrections automatiques appliquées ici sont conservatrices : suppression des doublons de lignes strictement identiques dans les copies de travail, normalisation des espaces/casse des e-mails, normalisation des statuts et conversion des champs numériques/dates pour les contrôles. Les doublons d'identifiants non identiques, les clients rapprochables et les valeurs manquantes ne sont pas supprimés ou imputés automatiquement.

In [5]:
avant_apres = []
copies = {nom: df.copy(deep=True) for nom, df in brut.items()}

for nom, df in copies.items():
    n_avant = len(df)
    df2 = df.drop_duplicates().copy()
    avant_apres.append({
        "table": nom, "operation": "Suppression des lignes strictement identiques",
        "lignes_avant": n_avant, "lignes_apres": len(df2),
        "lignes_touchees": n_avant-len(df2)
    })
    copies[nom] = df2

# Normalisations non destructives dans les copies de travail
if "email" in copies["clients"]:
    copies["clients"]["email_normalise"] = (
        copies["clients"]["email"].astype("string").str.strip().str.lower()
    )
if "statut" in copies["commandes"]:
    copies["commandes"]["statut_normalise"] = (
        copies["commandes"]["statut"].fillna("").astype(str).str.strip().str.lower()
        .map(lambda x: ''.join(ch for ch in unicodedata.normalize("NFKD", x)
                               if not unicodedata.combining(ch)))
    )

impact_dedoublonnage = pd.DataFrame(avant_apres)
display(impact_dedoublonnage)

# Les clés répétées après retrait des doublons strictement identiques restent à investiguer.
for table, cle in [("clients", "client_id"), ("commandes", "commande_id"), ("produits", "produit_id")]:
    df = copies[table]
    print(f"{table}: {df[cle].duplicated(keep=False).sum()} lignes concernées par une clé répétée après retrait des doublons exacts.")


,table,operation,lignes_avant,lignes_apres,lignes_touchees
0,clients,Suppression des lignes strictement identiques,4160,4160,0
1,commandes,Suppression des lignes strictement identiques,9583,9497,86
2,lignes,Suppression des lignes strictement identiques,16868,16787,81
3,produits,Suppression des lignes strictement identiques,120,120,0


clients: 0 lignes concernées par une clé répétée après retrait des doublons exacts.
commandes: 8 lignes concernées par une clé répétée après retrait des doublons exacts.
produits: 0 lignes concernées par une clé répétée après retrait des doublons exacts.


## 6. Cohérence commandes / lignes et chiffre d’affaires

In [6]:
# Calcul du sous-total des lignes : quantité x prix unitaire x (1 - remise/100)
lignes_calc = copies["lignes"].copy()
lignes_calc["quantite_num"] = pd.to_numeric(lignes_calc["quantite"], errors="coerce")
lignes_calc["prix_num"] = parse_montant(lignes_calc["prix_unitaire"])
lignes_calc["remise_num"] = parse_montant(lignes_calc["remise_pct"])
lignes_calc["montant_ligne"] = (
    lignes_calc["quantite_num"] * lignes_calc["prix_num"] *
    (1 - lignes_calc["remise_num"] / 100)
)
sous_totaux = lignes_calc.groupby("commande_id", dropna=False)["montant_ligne"].sum(min_count=1).rename("sous_total_lignes")

commandes_calc = copies["commandes"].copy()
commandes_calc["montant_num"] = parse_montant(commandes_calc["montant_ttc"])
commandes_calc["date_parsee"] = parse_date(commandes_calc["date_commande"])
commandes_calc["statut_normalise"] = (
    commandes_calc["statut"].fillna("").astype(str).str.strip().str.lower()
    .map(lambda x: ''.join(ch for ch in unicodedata.normalize("NFKD", x)
                           if not unicodedata.combining(ch)))
)
commandes_calc = commandes_calc.merge(sous_totaux, left_on="commande_id", right_index=True, how="left")

canal = commandes_calc["canal"].fillna("").astype(str).str.strip().str.lower()
frais = np.where(
    canal.isin(["web", "marketplace"]) & (commandes_calc["sous_total_lignes"].lt(SEUIL_LIVRAISON_GRATUITE).fillna(False)),
    FRAIS_LIVRAISON, 0.0
)
commandes_calc["montant_theorique"] = commandes_calc["sous_total_lignes"] + frais
commandes_calc["ecart_euros"] = commandes_calc["montant_num"] - commandes_calc["montant_theorique"]
commandes_calc["ecart_superieur_tolerance"] = commandes_calc["ecart_euros"].abs() > TOLERANCE_EURO

colonnes_reco = ["commande_id", "client_id", "date_commande", "canal", "statut",
                 "montant_num", "sous_total_lignes", "montant_theorique", "ecart_euros",
                 "ecart_superieur_tolerance"]
reconciliation = commandes_calc[colonnes_reco].copy()
display(reconciliation.head(15))

print("Commandes avec écart supérieur à 2 centimes :",
      int(reconciliation["ecart_superieur_tolerance"].fillna(False).sum()))
print("Commandes sans sous-total calculable :",
      int(reconciliation["sous_total_lignes"].isna().sum()))

# CA brut déclaré et CA net selon règle métier : seules les commandes livrées contribuent.
# Les commandes retournées sont remboursées et les annulées ne génèrent pas de CA.
montants = commandes_calc["montant_num"]
statut = commandes_calc["statut_normalise"]
ca = pd.DataFrame([
    {"indicateur": "Montant TTC déclaré — toutes commandes (non retraité)", "montant_eur": montants.sum(min_count=1)},
    {"indicateur": "CA net provisoire — commandes livrées uniquement", "montant_eur": montants[statut.eq("livree")].sum(min_count=1)},
    {"indicateur": "Montants des commandes annulées (exclus du CA)", "montant_eur": montants[statut.eq("annulee")].sum(min_count=1)},
    {"indicateur": "Montants des commandes retournées (remboursées)", "montant_eur": montants[statut.eq("retournee")].sum(min_count=1)},
])
display(ca)
print("Attention : CA provisoire calculé sur les montants déclarés ; les écarts de rapprochement et les commandes orphelines doivent être arbitrés avant validation.")


,commande_id,client_id,date_commande,canal,statut,montant_num,sous_total_lignes,montant_theorique,ecart_euros,ecart_superieur_tolerance
0,CMD-2022-000001,C003662,17/01/2022 13:33,marketplace,livree,35.8,30.9,35.8,0.0,False
1,CMD-2022-000002,C002639,2022-01-20,web,livree,182.52,182.52,182.52,-0.0,False
2,CMD-2022-000003,C000180,2022-01-28,web,livree,166.9,166.9,166.9,0.0,False
3,CMD-2022-000004,C000231,2022-02-03,marketplace,livree,387.5,387.5,387.5,0.0,False
4,CMD-2022-000005,C002967,2022-02-06,web,livree,155.52,155.52,155.52,0.0,False
5,CMD-2022-000006,C003031,2022-02-12,magasin,Livrée,220.7,220.7,220.7,-0.0,False
6,CMD-2022-000007,C000194,2022-02-21,magasin,livree,125.9,125.9,125.9,0.0,False
7,CMD-2022-000008,C001949,2022-02-22,web,livree,57.8,52.9,57.8,0.0,False
8,CMD-2022-000009,C002554,2022-02-22,magasin,livree,141.8,99.9,99.9,41.9,True
9,CMD-2022-000010,C002967,2022-02-27,web,livree,271.8,271.8,271.8,0.0,False


Commandes avec écart supérieur à 2 centimes : 279
Commandes sans sous-total calculable : 138


,indicateur,montant_eur
0,Montant TTC déclaré — toutes commandes (non re...,1832885.01
1,CA net provisoire — commandes livrées uniquement,1573171.91
2,Montants des commandes annulées (exclus du CA),144576.12
3,Montants des commandes retournées (remboursées),115136.98


Attention : CA provisoire calculé sur les montants déclarés ; les écarts de rapprochement et les commandes orphelines doivent être arbitrés avant validation.


## 7. Contrôles qualité automatiques (au moins 8)

In [7]:
# Les contrôles sont évalués et affichés ; les anomalies connues sont rapportées,
# elles ne font pas toutes échouer le notebook.
controle_auto = []

def ajouter_controle(nom, condition, detail):
    controle_auto.append({"contrôle": nom, "résultat": "OK" if bool(condition) else "À CORRIGER / INVESTIGUER",
                          "détail": detail})

ajouter_controle("Volumétrie clients non nulle", len(copies["clients"]) > 0, f"{len(copies['clients'])} lignes")
ajouter_controle("Volumétrie commandes non nulle", len(copies["commandes"]) > 0, f"{len(copies['commandes'])} lignes")
ajouter_controle("Clé produit unique après retrait exact", copies["produits"]["produit_id"].is_unique,
                 f"{copies['produits']['produit_id'].nunique()} clés distinctes")
ajouter_controle("Clé client unique après retrait exact", copies["clients"]["client_id"].is_unique,
                 f"{copies['clients']['client_id'].nunique()} clés distinctes")
ajouter_controle("Clé commande unique après retrait exact", copies["commandes"]["commande_id"].is_unique,
                 f"{copies['commandes']['commande_id'].nunique()} clés distinctes")
ajouter_controle("Aucune quantité <= 0", bool((lignes_calc["quantite_num"] > 0).all()),
                 f"{int((lignes_calc['quantite_num'] <= 0).sum())} lignes à quantité nulle/négative")
ajouter_controle("Remises comprises entre 0 et 100", bool(lignes_calc["remise_num"].between(0, 100).all()),
                 f"{int((~lignes_calc['remise_num'].between(0, 100)).sum())} valeurs hors bornes")
ajouter_controle("Dates de commande <= date de référence", bool((commandes_calc["date_parsee"].dropna() <= DATE_REFERENCE).all()),
                 f"{int((commandes_calc['date_parsee'] > DATE_REFERENCE).sum())} dates futures")
ajouter_controle("Toutes les lignes rattachées à une commande connue",
                 bool(lignes_calc["commande_id"].isin(copies["commandes"]["commande_id"]).all()),
                 f"{int((~lignes_calc['commande_id'].isin(copies['commandes']['commande_id'])).sum())} références inconnues")
ajouter_controle("Toutes les lignes rattachées à un produit connu",
                 bool(lignes_calc["produit_id"].isin(copies["produits"]["produit_id"]).all()),
                 f"{int((~lignes_calc['produit_id'].isin(copies['produits']['produit_id'])).sum())} références inconnues")
ajouter_controle("Montants de lignes calculables",
                 bool(lignes_calc["montant_ligne"].notna().all()),
                 f"{int(lignes_calc['montant_ligne'].isna().sum())} lignes non calculables")
ajouter_controle("Écarts de réconciliation <= 2 centimes",
                 bool((reconciliation["ecart_euros"].dropna().abs() <= TOLERANCE_EURO).all()),
                 f"{int(reconciliation['ecart_superieur_tolerance'].fillna(False).sum())} commandes avec écart")

rapport_controles = pd.DataFrame(controle_auto)
display(rapport_controles)


,contrôle,résultat,détail
0,Volumétrie clients non nulle,OK,4160 lignes
1,Volumétrie commandes non nulle,OK,9497 lignes
2,Clé produit unique après retrait exact,OK,120 clés distinctes
3,Clé client unique après retrait exact,OK,4160 clés distinctes
4,Clé commande unique après retrait exact,À CORRIGER / INVESTIGUER,9493 clés distinctes
5,Aucune quantité <= 0,À CORRIGER / INVESTIGUER,51 lignes à quantité nulle/négative
6,Remises comprises entre 0 et 100,OK,0 valeurs hors bornes
7,Dates de commande <= date de référence,À CORRIGER / INVESTIGUER,47 dates futures
8,Toutes les lignes rattachées à une commande co...,OK,0 références inconnues
9,Toutes les lignes rattachées à un produit connu,À CORRIGER / INVESTIGUER,102 références inconnues


## 8. Synthèse des anomalies et recommandations
À compléter après lecture des tableaux précédents. Ne pas présenter les anomalies comme des corrections définitives sans validation métier.

**Principes de décision**
- Doublons exacts : suppression possible dans une copie, avec mesure du nombre de lignes retirées.
- Identifiants répétés avec contenus différents : conserver pour investigation ; ne pas supprimer automatiquement.
- E-mails répétés : signal de rapprochement, pas preuve suffisante qu'il s'agit de la même personne.
- Valeurs manquantes : ne pas imputer sans règle métier documentée.
- Quantités négatives/nulles, dates futures et montants négatifs : isoler et faire valider.
- Commandes annulées : exclues du chiffre d’affaires ; commandes retournées : remboursées.
- Écarts montant commande / somme des lignes : analyser les cas, les frais de livraison, les retours et les anomalies de lignes avant correction.


## 9. Export des livrables de l’auditrice qualité

In [8]:
DOSSIER_SORTIE = Path("sorties_audit")
DOSSIER_SORTIE.mkdir(exist_ok=True)

# Export des tableaux de contrôle
profil_structure.to_csv(DOSSIER_SORTIE / "01_volumetrie.csv", index=False, encoding="utf-8-sig")
audit_manquants.to_csv(DOSSIER_SORTIE / "02_valeurs_manquantes.csv", index=False, encoding="utf-8-sig")
controle_cles.to_csv(DOSSIER_SORTIE / "03_integrite_referentielle.csv", index=False, encoding="utf-8-sig")
audit_validite.to_csv(DOSSIER_SORTIE / "04_anomalies_validite.csv", index=False, encoding="utf-8-sig")
impact_dedoublonnage.to_csv(DOSSIER_SORTIE / "05_impact_corrections.csv", index=False, encoding="utf-8-sig")
reconciliation.to_csv(DOSSIER_SORTIE / "06_reconciliation_commandes.csv", index=False, encoding="utf-8-sig")
ca.to_csv(DOSSIER_SORTIE / "07_synthese_chiffre_affaires.csv", index=False, encoding="utf-8-sig")
rapport_controles.to_csv(DOSSIER_SORTIE / "08_controles_automatiques.csv", index=False, encoding="utf-8-sig")

print("Livrables exportés dans :", DOSSIER_SORTIE.resolve())
print("Fichiers :", *sorted(p.name for p in DOSSIER_SORTIE.glob("*.csv")), sep="\n- ")


Livrables exportés dans : /home/oai/sorties_audit
Fichiers :
- 01_volumetrie.csv
- 02_valeurs_manquantes.csv
- 03_integrite_referentielle.csv
- 04_anomalies_validite.csv
- 05_impact_corrections.csv
- 06_reconciliation_commandes.csv
- 07_synthese_chiffre_affaires.csv
- 08_controles_automatiques.csv


## 10. Conclusion — rôle d’AKBI Hayat
En tant qu’auditrice qualité, j’ai évalué la complétude, la validité, l’unicité, la cohérence et l’intégrité référentielle des quatre fichiers. J’ai mesuré l’effet des suppressions de doublons strictement identiques dans des copies, identifié les anomalies qui nécessitent une décision métier et rapproché les montants déclarés des commandes avec les montants recalculés à partir des lignes et des frais de livraison. Les données brutes ont été conservées intactes.

**Limite importante :** cet audit ne constitue pas à lui seul la validation finale du chiffre d’affaires. Les écarts, les commandes sans client connu, les commandes sans lignes calculables et les statuts ambigus doivent être arbitrés avec l’équipe avant toute correction définitive.
